# デモ2：言葉で探す（OWLv2）

探したい物の名前を言葉で渡すと、画像のどこにあるかを枠で示します。物体検出のモデルは、ふつう決まったクラス（「人」「車」など80種類）しか探せませんが、このモデルは**学習していない物の名前でも探せます**。

- モデル：[google/owlv2-base-patch16-ensemble](https://huggingface.co/google/owlv2-base-patch16-ensemble)（Apache-2.0）
- 中の Transformer：画像を 16×16 のパッチに分けて読む ViT。各パッチのトークンから、枠の位置と「その枠に写っている物の特徴」を出し、文の特徴と比べる

**最初に**：「ファイル」→「ドライブにコピーを保存」で自分のコピーを作り、「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行します。コードを読む必要はありません。何ができて、何が苦手かを見つけるのが目的です。

## 1. 準備

目安：約2分（うち計算 約1分）

In [ ]:
import importlib.util
import subprocess
import sys

# Colab に入っていない、または古いライブラリを入れる
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import data

device = "cuda" if torch.cuda.is_available() else "cpu"
# T4 は bfloat16 に対応していないので、GPU では float16 を使う
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

In [ ]:
from transformers import pipeline

detector = pipeline("zero-shot-object-detection", model="google/owlv2-base-patch16-ensemble",
                    device=device, dtype=dtype)


def show_detections(image, results, title=""):
    """枠とラベルを描く"""
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)
    for r in results:
        b = r["box"]
        ax.add_patch(plt.Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"],
                                   fill=False, color="red", linewidth=2))
        ax.text(b["xmin"], b["ymin"], f"{r['label']} {r['score']:.2f}", color="white",
                backgroundcolor="red", fontsize=9)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

## 2. 例の画像で試す

目安：約3分

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる（JPEG か PNG）

# 例の画像（scikit-image に入っている写真。CC0 またはパブリックドメイン）
images = {
    "astronaut": Image.fromarray(data.astronaut()),
    "cat": Image.fromarray(data.chelsea()),
    "coffee": Image.fromarray(data.coffee()),
}
if USE_UPLOAD:
    import io
    from google.colab import files
    for name, content in files.upload().items():
        images[name] = Image.open(io.BytesIO(content)).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, (name, image) in zip(axes[0], images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

探す物の名前（`QUERIES`）を英語で書きます。`THRESHOLD` より低いスコアの枠は表示しません。

In [ ]:
QUERIES = ["a face", "a flag", "a cat's ear", "a cup", "a spoon"]  # ← 変えてみる
THRESHOLD = 0.2  # ← 変えてみる（下げると枠が増え、まちがいも増える）

for name, image in images.items():
    results = detector(image, candidate_labels=QUERIES, threshold=THRESHOLD)
    show_detections(image, results, name)

## 3. やってみよう

目安：約10分

- `QUERIES` に、細かい部分（`"an eye"`、`"a whisker"`）や、色付きの指定（`"a red object"`）を入れます。
- `THRESHOLD` を 0.1、0.3 に変えて、枠の数とまちがいの関係を見ます。
- 研究室や身の回りの写真をアップロードして、研究で探したい物（部品、傷、文字など）を言葉で探せるか試します。